# デバイスインターフェース 動作確認ノートブック

このノートブックでは、MADSciとは独立して各デバイスインターフェースの動作を確認します。

**フェイクインターフェース**: ハードウェア不要。いつでも実行可能。  
**実インターフェース**: COMポートに実機を接続した状態で実行。

In [1]:
import sys
import os

# devices/ ディレクトリをパスに追加
sys.path.insert(0, os.path.abspath(".."))

# インポート確認
from devices.balance_fake_interface import BalanceFakeInterface
from devices.high_viscosity_dispenser_fake_interface import HighViscosityDispenserFakeInterface

print("インポート成功")

インポート成功


---
## 1. 天秤 (BalanceFakeInterface)

In [2]:
# 天秤フェイクインターフェースを初期化
balance = BalanceFakeInterface(
    latency=0.1,       # 実機の0.1倍の速さで動作
    base_mass_g=0.0,   # 初期質量
    noise_std=0.01,    # ノイズ (標準偏差 g)
    failure_rate=0.0,  # エラー発生率 (0.0 〜 1.0)
)
print(f"status: {balance.status}")

status: connected


In [3]:
# tare (風袋引き)
balance.tare()
print(f"tare後の質量: {balance.current_mass_g} g")

tare後の質量: 0.0 g


In [4]:
# 重量読み取り (settle_time=0 で即時読み取り)
weight = balance.read_weight(settle_time=0)
print(f"read_weight: {weight} g")

read_weight: -0.0025 g


In [5]:
# 複数回読み取って、ノイズのばらつきを確認
readings = [balance.read_weight(settle_time=0) for _ in range(10)]
print("10回読み取り結果:")
for i, r in enumerate(readings):
    print(f"  [{i+1}] {r:.4f} g")

10回読み取り結果:
  [1] -0.0016 g
  [2] -0.0130 g
  [3] -0.0128 g
  [4] -0.0220 g
  [5] -0.0073 g
  [6] -0.0194 g
  [7] -0.0186 g
  [8] -0.0207 g
  [9] -0.0290 g
  [10] -0.0306 g


In [6]:
balance.close()
print(f"status: {balance.status}")

status: disconnected


---
## 2. 高粘度ディスペンサー (HighViscosityDispenserFakeInterface)

In [7]:
# ディスペンサーフェイクインターフェースを初期化
dispenser = HighViscosityDispenserFakeInterface(
    full_steps_per_rev=200,
    microstep_multiplier=1,
    purge_speed_rps=0.5,
    latency=0.1,
    failure_rate=0.0,
)
print(f"status: {dispenser.status}")
print(f"motion_status: {dispenser.motion_status}")

status: connected
motion_status: idle


In [8]:
# パージ (ノズル充填)
dispenser.purge(rotations=1.0)
print(f"motion_status after purge: {dispenser.motion_status}")

motion_status after purge: idle


In [9]:
# ディスペンス (吐出)
dispenser.dispense(rotations=2.0, speed_rps=1.0)
print(f"motion_status after dispense: {dispenser.motion_status}")

motion_status after dispense: idle


In [10]:
# サックバック (液切り)
dispenser.suck_back(rotations=0.2, speed_rps=0.5)
print(f"motion_status after suck_back: {dispenser.motion_status}")

motion_status after suck_back: idle


In [11]:
dispenser.close()
print(f"status: {dispenser.status}")

status: disconnected


---
## 3. 実機テスト（COMポート接続時のみ）

実機を接続した状態でのみ以下のセルを実行してください。

In [13]:
# 接続されているCOMポートを確認
import serial.tools.list_ports

ports = list(serial.tools.list_ports.comports())
if ports:
    for p in ports:
        print(f"{p.device}: {p.description}")
else:
    print("COMポートが見つかりません（実機未接続）")

COM4: Bluetooth リンク経由の標準シリアル (COM4)
COM5: Bluetooth リンク経由の標準シリアル (COM5)
COM3: Intel(R) Active Management Technology - SOL (COM3)


In [14]:
# 天秤 実機テスト
# COM_PORT = "COM3"  # 実際のCOMポートに変更してください

# from devices.balance_interface import BalanceInterface
# balance_real = BalanceInterface(port=COM_PORT)
# print(balance_real.read_weight())
# balance_real.close()

In [15]:
# ディスペンサー 実機テスト
# COM_PORT = "COM4"  # 実際のCOMポートに変更してください

# from devices.high_viscosity_dispenser_interface import HighViscosityDispenserInterface
# dispenser_real = HighViscosityDispenserInterface(
#     port=COM_PORT,
#     full_steps_per_rev=200,
#     microstep_multiplier=1,
#     purge_speed_rps=0.5,
# )
# dispenser_real.purge(rotations=1.0)
# dispenser_real.close()